In [38]:
# ==============================================================================
# Author      : Prajwal Ingle
# GitHub      : https://github.com/Prajwal-Ingle
# Contact     : Prajwalingle80800@gmail.com
# Project     : Text-to-SQL Fine-Tuning using Pure LoRA (FP16)
# Base Model  : Qwen/Qwen2.5-Coder-1.5B-Instruct
# Dataset     : b-mc2/sql-create-context (3,000-row sample: 2,850 train / 150 val)
# Environment : Kaggle Notebook (Tesla T4 GPU, FP16)
# ==============================================================================

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [6]:
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6"

In [7]:
import sys
import torch

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("CUDA avail :", torch.cuda.is_available())
print("GPU count  :", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory : {props.total_memory / 1024**3:.2f} GB")

Python     : 3.13.16
PyTorch    : 2.11.0+cu130
CUDA avail : True
GPU count  : 1
GPU name   : Tesla T4
GPU memory : 14.56 GB


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map={"": 0},
)

model.config.use_cache = False
if hasattr(model.config, "pretraining_tp"):
    model.config.pretraining_tp = 1

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params / 1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total parameters: 1543.7 M
Memory footprint: 2.88 GB


In [22]:
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [23]:
PROBES = [
    {
        "schema": "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question": "List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema": "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question": "What is the total order amount per customer in 2024?",
    },
    {
        "schema": "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question": "Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("=" * 70)
print("BASE MODEL (before fine-tuning)")
print("=" * 70)
base_outputs = []
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASE MODEL (before fine-tuning)

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: SELECT customer_id, SUM(amount) as total_order_amount 
FROM orders 
WHERE YEAR(order_date) = 2024 
GROUP BY customer_id;

--- Probe 3 ---
Q: Show the top 5 highest rated horror movies released after 2015.
A: SELECT title, rating FROM movies WHERE genre = 'Horror' AND YEAR > 2015 ORDER BY rating DESC LIMIT 5


In [24]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context", split="train")
raw = raw.shuffle(seed=42).select(range(3000))
split = raw.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Train:", len(train_ds), " Eval:", len(eval_ds))


def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant = row["answer"]
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}


train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(format_example, remove_columns=eval_ds.column_names)

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Train: 2850  Eval: 150


Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [25]:
from peft import LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [26]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/qwen2.5-coder-sql-lora-checkpoints"
ADAPTER_DIR = "/kaggle/working/qwen2.5-coder-1.5b-sql-lora-adapter"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )


train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

trainer.train()
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.796048,0.761358
100,0.726887,0.700852
150,0.686972,0.686556


Peak GPU memory allocated: 3.76 GB


In [27]:
model.config.use_cache = True
model.eval()

print("=" * 70)
print("FINE-TUNED MODEL (after LoRA)")
print("=" * 70)
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i - 1])
    print("=" * 70)
    print("AFTER :", ans)
    print("=" * 70)

FINE-TUNED MODEL (after LoRA)

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: SELECT name FROM employees WHERE department = 'Engineering' AND salary > 100000
AFTER : SELECT name FROM employees WHERE department = "Engineering" AND salary > 100000

--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: SELECT customer_id, SUM(amount) as total_order_amount 
FROM orders 
WHERE YEAR(order_date) = 2024 
GROUP BY customer_id;
AFTER : SELECT SUM(amount) AS total_amount FROM orders WHERE order_date BETWEEN "2024-01-01" AND "2024-12-31" GROUP BY customer_id

--- Probe 3 ---
Q:      Show the top 5 highest rated horror movies released after 2015.
BEFORE: SELECT title, rating FROM movies WHERE genre = 'Horror' AND YEAR > 2015 ORDER BY rating DESC LIMIT 5
AFTER : SELECT title FROM movies WHERE rating > 8 AND genre = "Horror" AND year > 2015 ORDER BY rating DESC LIMIT 5


In [28]:
import os

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

total_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
) / (1024**2)
print(f"Adapter size on disk: {total_mb:.2f} MB at {ADAPTER_DIR}")

Adapter size on disk: 27.56 MB at /kaggle/working/qwen2.5-coder-1.5b-sql-lora-adapter


In [37]:
import re
import sqlite3
from collections import Counter
from datasets import load_dataset
import pandas as pd

# Load raw held-out eval split (has 'context', 'question', 'answer')
raw_eval_ds = (
    load_dataset("b-mc2/sql-create-context", split="train")
    .shuffle(seed=42)
    .select(range(3000))
    .train_test_split(test_size=0.05, seed=42)["test"]
)


def clean_sql_output(text: str) -> tuple[str, bool]:
    raw_out = text.strip()
    has_fence = "```" in raw_out
    sql = re.sub(r"^```(?:sql)?\s*", "", raw_out, flags=re.IGNORECASE)
    sql = re.sub(r"\s*```$", "", sql).strip()
    if ";" in sql:
        sql = sql.split(";")[0].strip() + ";"
    return sql, (not has_fence)


def run_on_mock_db(create_sql: str, gold_sql: str, pred_sql: str) -> tuple[bool, bool]:
    """Returns (pred_executed_ok, results_matched)."""
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    try:
        ddl = re.sub(r"\b(VARCHAR|NVARCHAR|CHAR|NCHAR|TEXT)\b", "TEXT", create_sql, flags=re.I)
        ddl = re.sub(r"\b(INT|INTEGER|BIGINT|SMALLINT|TINYINT)\b", "INTEGER", ddl, flags=re.I)
        ddl = re.sub(r"\b(FLOAT|DOUBLE|REAL|NUMERIC|DECIMAL)\b", "REAL", ddl, flags=re.I)
        cur.executescript(ddl)

        tables = cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table'").fetchall()
        for tbl_name, tbl_sql in tables:
            m = re.search(r"\((.*)\)", tbl_sql, flags=re.S)
            if not m:
                continue
            cols, row1, row2 = [], [], []
            for idx, col_def in enumerate(m.group(1).split(",")):
                parts = col_def.strip().split()
                if not parts:
                    continue
                cname = parts[0].strip('"`[]')
                if cname.upper() in {"PRIMARY", "FOREIGN", "UNIQUE", "CHECK", "CONSTRAINT"}:
                    continue
                ctype = " ".join(parts[1:]).upper()
                cols.append(cname)
                if "INT" in ctype:
                    row1.append(2024 if "YEAR" in cname.upper() else (idx + 1) * 10)
                    row2.append(2018 if "YEAR" in cname.upper() else (idx + 1) * 20)
                elif any(t in ctype for t in ["REAL", "FLOAT", "DOUBLE", "NUM", "DEC"]):
                    row1.append(9.2 if "RATING" in cname.upper() else 120.0)
                    row2.append(6.5 if "RATING" in cname.upper() else 50.0)
                else:
                    row1.append("engineering" if "DEPT" in cname.upper() else f"val_{cname}_1")
                    row2.append("sales" if "DEPT" in cname.upper() else f"val_{cname}_2")

            if cols:
                q_cols = ",".join(f'"{c}"' for c in cols)
                ph = ",".join(["?"] * len(cols))
                ins = f'INSERT INTO "{tbl_name}" ({q_cols}) VALUES ({ph})'
                cur.execute(ins, row1)
                cur.execute(ins, row2)
        conn.commit()

        gold_rows = Counter(
            tuple(round(v, 6) if isinstance(v, float) else v for v in r)
            for r in cur.execute(gold_sql).fetchall()
        )
        pred_rows = Counter(
            tuple(round(v, 6) if isinstance(v, float) else v for v in r)
            for r in cur.execute(pred_sql).fetchall()
        )
        conn.close()
        return True, (gold_rows == pred_rows)
    except Exception:
        conn.close()
        return False, False


def test_accuracy(dataset, name: str, limit: int = 150) -> dict:
    model.eval()
    model.config.use_cache = True

    tested = valid = clean = matched = 0
    for i in range(min(len(dataset), limit)):
        row = dataset[i]
        # Skip rows whose gold SQL isn't valid SQLite
        ok_gold, _ = run_on_mock_db(row["context"], row["answer"], row["answer"])
        if not ok_gold:
            continue

        tested += 1
        prompt = build_prompt(row["context"], row["question"])
        raw_ans = generate(prompt, max_new_tokens=128)
        pred_sql, is_clean = clean_sql_output(raw_ans)
        if is_clean:
            clean += 1

        ok_pred, is_match = run_on_mock_db(row["context"], row["answer"], pred_sql)
        if ok_pred:
            valid += 1
            if is_match:
                matched += 1

    return {
        "Model": name,
        "Tested": tested,
        "Execution Acc (%)": round(100 * matched / max(tested, 1), 2),
        "Valid SQL (%)": round(100 * valid / max(tested, 1), 2),
        "Clean Format (%)": round(100 * clean / max(tested, 1), 2),
    }


with model.disable_adapter():
    base_row = test_accuracy(raw_eval_ds, "Base (No LoRA)", limit=150)

ft_row = test_accuracy(raw_eval_ds, "Fine-Tuned (+LoRA)", limit=150)
display(pd.DataFrame([base_row, ft_row]))

,Model,Tested,Execution Acc (%),Valid SQL (%),Clean Format (%)
0,Base (No LoRA),143,81.82,91.61,100.0
1,Fine-Tuned (+LoRA),143,90.91,98.60,100.0
